In [6]:
# ============================================================
# QLoRA FINETUNING
# Qwen2.5-Coder-3B-Instruct
# A2UI Dataset
# Kaggle T4/P100
# ============================================================

import os
import shutil
from pathlib import Path

from datasets import load_dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import kagglehub


# ============================================================
# CONFIG
# ============================================================

MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit"

MAX_SEQ_LEN = 1536

OUTPUT_DIR = "/kaggle/working/qwen25_a2ui_lora"

# Your actual Kaggle dataset paths
TRAIN_FILE = "/kaggle/input/datasets/rumanakhaisarp/llm-ui-generator/train.jsonl"
VAL_FILE   = "/kaggle/input/datasets/rumanakhaisarp/llm-ui-generator/val.jsonl"

# Your Kaggle username
KAGGLE_USERNAME = "rumanakhaisarp"

# Permanent Kaggle Dataset
DATASET_SLUG = "a2ui-finetuned-gguf"

KAGGLE_DATASET = f"{KAGGLE_USERNAME}/{DATASET_SLUG}"

# Start a fresh training run
RESUME_FROM_CHECKPOINT = None


# ============================================================
# DIRECTORIES
# ============================================================

ADAPTER_DIR = os.path.join(
    OUTPUT_DIR,
    "final_adapter"
)

GGUF_DIR = os.path.join(
    OUTPUT_DIR,
    "gguf_export"
)

UPLOAD_DIR = "/kaggle/working/a2ui_gguf_upload"


os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

# Remove old upload directory if it exists
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)

os.makedirs(UPLOAD_DIR, exist_ok=True)


# ============================================================
# CHECK GPU
# ============================================================

import torch

print("=" * 70)
print("GPU INFORMATION")
print("=" * 70)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3),
            2
        ),
        "GB"
    )

else:
    raise RuntimeError(
        "CUDA GPU not available. Enable a Kaggle GPU accelerator."
    )


# ============================================================
# CHECK DATASET FILES
# ============================================================

print("=" * 70)
print("CHECKING DATASET")
print("=" * 70)

print("Train:", TRAIN_FILE)
print("Validation:", VAL_FILE)

if not os.path.exists(TRAIN_FILE):
    raise FileNotFoundError(
        f"Training file not found:\n{TRAIN_FILE}"
    )

if not os.path.exists(VAL_FILE):
    raise FileNotFoundError(
        f"Validation file not found:\n{VAL_FILE}"
    )

print("Training file found.")
print("Validation file found.")


# ============================================================
# LOAD MODEL
# ============================================================

print("=" * 70)
print("LOADING QWEN2.5-CODER-3B")
print("=" * 70)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)

print("Model loaded successfully.")


# ============================================================
# APPLY LoRA
# ============================================================

print("=" * 70)
print("CONFIGURING LoRA")
print("=" * 70)

model = FastLanguageModel.get_peft_model(
    model,

    r=16,

    lora_alpha=32,

    lora_dropout=0.0,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],

    bias="none",

    use_gradient_checkpointing="unsloth",

    random_state=42,
)

print("LoRA configured successfully.")


# ============================================================
# LOAD DATASET
# ============================================================

print("=" * 70)
print("LOADING DATASET")
print("=" * 70)

dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "val": VAL_FILE,
    }
)

print("Training examples:", len(dataset["train"]))
print("Validation examples:", len(dataset["val"]))


# ============================================================
# FORMAT DATASET
# ============================================================

print("=" * 70)
print("FORMATTING CHAT DATA")
print("=" * 70)


def formatting_func(example):

    messages = [
        {
            "role": "system",
            "content": example["system"],
        },
        {
            "role": "user",
            "content": example["prompt"],
        },
        {
            "role": "assistant",
            "content": example["completion"],
        },
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


dataset = dataset.map(
    lambda example: {
        "text": formatting_func(example)
    }
)

print("Formatting completed.")


# ============================================================
# SHOW SAMPLE
# ============================================================

print("=" * 70)
print("SAMPLE TRAINING EXAMPLE")
print("=" * 70)

print(
    dataset["train"][0]["text"][:2000]
)


# ============================================================
# TRAINER
# ============================================================

print("=" * 70)
print("CREATING SFT TRAINER")
print("=" * 70)

trainer = SFTTrainer(
    model=model,

    processing_class=tokenizer,

    train_dataset=dataset["train"],

    eval_dataset=None,

    args=SFTConfig(

        # -------------------------
        # Output
        # -------------------------

        output_dir=OUTPUT_DIR,

        # -------------------------
        # Dataset
        # -------------------------

        dataset_text_field="text",

        max_length=MAX_SEQ_LEN,

        # IMPORTANT:
        # packing MUST be inside SFTConfig
        packing=True,

        packing_strategy="bfd",

        # -------------------------
        # Batch
        # -------------------------

        per_device_train_batch_size=2,

        gradient_accumulation_steps=8,

        # Effective batch size:
        # 2 × 8 = 16

        # -------------------------
        # Training
        # -------------------------

        num_train_epochs=1,

        learning_rate=2e-4,

        warmup_ratio=0.05,

        # -------------------------
        # Logging
        # -------------------------

        logging_steps=10,

        # -------------------------
        # Checkpoints
        # -------------------------

        save_strategy="steps",

        save_steps=50,

        save_total_limit=3,

        # -------------------------
        # Precision
        # -------------------------

        fp16=True,

        bf16=False,

        # -------------------------
        # Optimizer
        # -------------------------

        optim="adamw_8bit",

        # -------------------------
        # Misc
        # -------------------------

        report_to="none",

        seed=42,

        # Don't run evaluation
        eval_strategy="no",
    ),
)

print("Trainer created successfully.")


# ============================================================
# START TRAINING
# ============================================================

print("=" * 70)
print("STARTING QLORA TRAINING")
print("=" * 70)

trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)

print("=" * 70)
print("TRAINING FINISHED")
print("=" * 70)


# ============================================================
# SAVE LoRA ADAPTER
# ============================================================

print("=" * 70)
print("SAVING LoRA ADAPTER")
print("=" * 70)

model.save_pretrained(
    ADAPTER_DIR
)

tokenizer.save_pretrained(
    ADAPTER_DIR
)

print(
    "Adapter saved to:"
)

print(
    ADAPTER_DIR
)


# ============================================================
# EXPORT Q4_K_M GGUF
# ============================================================

print("=" * 70)
print("EXPORTING Q4_K_M GGUF")
print("=" * 70)

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method="q4_k_m",
)

print(
    "GGUF export completed."
)

print(
    "GGUF directory:",
    GGUF_DIR
)


# ============================================================
# FIND GGUF FILE
# ============================================================

print("=" * 70)
print("SEARCHING FOR GGUF")
print("=" * 70)

gguf_files = list(
    Path(GGUF_DIR).rglob("*.gguf")
)

if len(gguf_files) == 0:

    raise FileNotFoundError(
        "No GGUF file was found after conversion."
    )


for gguf_file in gguf_files:

    size_gb = (
        gguf_file.stat().st_size
        / (1024 ** 3)
    )

    print(
        f"GGUF: {gguf_file}"
    )

    print(
        f"Size: {size_gb:.2f} GB"
    )


# ============================================================
# COPY GGUF TO PERMANENT UPLOAD FOLDER
# ============================================================

print("=" * 70)
print("PREPARING PERMANENT KAGGLE DATASET")
print("=" * 70)

for gguf_file in gguf_files:

    destination = (
        Path(UPLOAD_DIR)
        / gguf_file.name
    )

    shutil.copy2(
        gguf_file,
        destination
    )

    print(
        "Copied:",
        gguf_file.name
    )


# ============================================================
# CREATE README
# ============================================================

README = """
# A2UI Qwen2.5-Coder-3B GGUF

Fine-tuned Qwen2.5-Coder-3B-Instruct model
for A2UI / UI generation.

## Base Model

unsloth/Qwen2.5-Coder-3B-Instruct-bnb-4bit

## Fine-tuning

QLoRA

LoRA rank: 16

LoRA alpha: 32

Epochs: 1

Maximum sequence length: 1536

## Quantization

Q4_K_M

## Dataset

A2UI LLM UI Generator dataset.
"""


with open(
    os.path.join(
        UPLOAD_DIR,
        "README.md"
    ),
    "w",
    encoding="utf-8",
) as f:

    f.write(README)


# ============================================================
# VERIFY UPLOAD FILES
# ============================================================

print("=" * 70)
print("FILES READY FOR PERMANENT UPLOAD")
print("=" * 70)

for file in Path(UPLOAD_DIR).iterdir():

    size_mb = (
        file.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{file.name:60s}"
        f"{size_mb:10.1f} MB"
    )


# ============================================================
# UPLOAD TO PERMANENT KAGGLE DATASET
# ============================================================

print("=" * 70)
print("UPLOADING TO PERMANENT KAGGLE DATASET")
print("=" * 70)

print(
    "Dataset:",
    KAGGLE_DATASET
)

try:

    kagglehub.dataset_upload(
        KAGGLE_DATASET,
        UPLOAD_DIR,
        version_notes=(
            "Qwen2.5-Coder-3B A2UI "
            "QLoRA Q4_K_M GGUF"
        ),
    )

    print("=" * 70)
    print("PERMANENT UPLOAD SUCCESSFUL")
    print("=" * 70)

    print(
        "Dataset:"
    )

    print(
        f"https://www.kaggle.com/datasets/"
        f"{KAGGLE_DATASET}"
    )

except Exception as e:

    print("=" * 70)
    print("WARNING: DATASET UPLOAD FAILED")
    print("=" * 70)

    print(
        "GGUF is still present locally at:"
    )

    print(
        GGUF_DIR
    )

    print(
        "Upload error:"
    )

    print(
        repr(e)
    )


# ============================================================
# FINAL SANITY CHECK
# ============================================================

print("=" * 70)
print("FINAL MODEL SANITY CHECK")
print("=" * 70)

FastLanguageModel.for_inference(model)

sample = dataset["val"][0]

inputs = tokenizer(
    sample["prompt"],
    return_tensors="pt",
).to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=1500,
    temperature=0.3,
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print(response)

print("=" * 70)
print("ALL STEPS COMPLETED")
print("=" * 70)

GPU INFORMATION
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB
CHECKING DATASET
Train: /kaggle/input/datasets/rumanakhaisarp/llm-ui-generator/train.jsonl
Validation: /kaggle/input/datasets/rumanakhaisarp/llm-ui-generator/val.jsonl
Training file found.
Validation file found.
LOADING QWEN2.5-CODER-3B
==((====))==  Unsloth 2026.8.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Model loaded successfully.
CONFIGURING LoRA
LoRA configured successfully.
LOADING DATASET
Training examples: 5302
Validation examples: 279
FORMATTING CHAT DATA


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Formatting completed.
SAMPLE TRAINING EXAMPLE
<|im_start|>system
Convert a user query + structured response into a rich A2UI v0.9 JSON UI array. Output ONLY the raw JSON array. No markdown, no explanation, no code fences.<|im_end|>
<|im_start|>user
Category: CREATIVE WRITING
Style: notion — Notion: clean white bg, warm gray surface, blue link accent #2383E2, tag Chips, ghost buttons, monospace code text, minimal borders

User Query: Create a 2-minute video script for a health tech tool ad targeting 25-60-year-old healthcare professionals on YouTube — including a tutorial and a 20% off promo code, with a professional tone and 1 patient outcome, and a $250 budget, and featuring a doctor endorsement in a hospital in New York

Structured Response:
## MediConnect Pro - YouTube Ad Preview

This structured preview card outlines a 2-minute YouTube ad script for "MediConnect Pro," a health tech tool designed to streamline practice management and enhance patient care for healthcare professionals

...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:27<00:00, 13.64s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:39<00:00, 19.81s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/qwen25_a2ui_lora/gguf_export`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10327-mix-b1e9972 (app-b10327-mix-b1e9972-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['/kaggle/working/qwen25_a2ui_lora/gguf_export_gguf/qwen2.5-coder-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model fi

FileNotFoundError: No GGUF file was found after conversion.

In [4]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --upgrade trl peft accelerate bitsandbytes kagglehub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 106.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 93.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 96.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes

In [ ]:
import unsloth
print(unsloth.__version__)

In [ ]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("llm-ui-generator")

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
os.environ["llm-ui-generator"] = UserSecretsClient().get_secret("llm-ui-generator")

In [8]:
import subprocess
gguf_path = "/kaggle/working/qwen25_a2ui_lora/gguf_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf"
result = subprocess.run(
    ["curl", "-v", "--upload-file", gguf_path,
     "https://transfer.sh/qwen2.5-coder-3b-instruct.Q4_K_M.gguf"],
    capture_output=True, text=True,
)
print("RETURN CODE:", result.returncode)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[-2000:])   # curl's verbose log goes to stderr

RETURN CODE: 7
STDOUT: 
STDERR:   % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0*   Trying 144.76.136.153:443...
* connect to 144.76.136.153 port 443 failed: Connection refused
*   Trying 2a01:4f8:200:1097::2:443...
* Immediate connect fail for 2a01:4f8:200:1097::2: Network is unreachable
* Failed to connect to transfer.sh port 443 after 96 ms: Connection refused

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
* Closing connection 0
curl: (7) Failed to connect to transfer.sh port 443 after 96 ms: Connection refused



In [9]:
result = subprocess.run(
    ["curl", "-F", f"file=@{gguf_path}", "https://0x0.st"],
    capture_output=True, text=True,
)
print(result.stdout, result.stderr)

uploads disabled because it’s been almost nothing but AI botnet spam for the past few months. will be back with a few changes at some point. no ETA.   % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0 1840M  100   150    0 65535    220  96118  5:34:38 --:--:--  5:34:38 96453



In [10]:
from IPython.display import FileLink

FileLink('/kaggle/working/qwen25_a2ui_lora/gguf_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf')

/kaggle/working/qwen25_a2ui_lora/gguf_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf

In [11]:
!split -b 500M "/kaggle/working/qwen25_a2ui_lora/gguf_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf" "/kaggle/working/qwen2.5_q4km.part_"

In [12]:
!ls -lh /kaggle/working/qwen2.5_q4km.part_*

-rw-r--r-- 1 root root 500M Aug  8 23:43 /kaggle/working/qwen2.5_q4km.part_aa
-rw-r--r-- 1 root root 500M Aug  8 23:43 /kaggle/working/qwen2.5_q4km.part_ab
-rw-r--r-- 1 root root 500M Aug  8 23:43 /kaggle/working/qwen2.5_q4km.part_ac
-rw-r--r-- 1 root root 341M Aug  8 23:43 /kaggle/working/qwen2.5_q4km.part_ad


In [15]:
parts = [
    "/kaggle/working/qwen2.5_q4km.part_aa",
    "/kaggle/working/qwen2.5_q4km.part_ab",
    "/kaggle/working/qwen2.5_q4km.part_ac",
    "/kaggle/working/qwen2.5_q4km.part_ad"
]

output = "/kaggle/working/qwen2.5-coder-3b-instruct.Q4_K_M.gguf"

with open(output, "wb") as outfile:
    for part in parts:
        with open(part, "rb") as infile:
            while True:
                chunk = infile.read(1024 * 1024)
                if not chunk:
                    break
                outfile.write(chunk)

print("Done!")

Done!


In [16]:
import os

path = "/kaggle/working/qwen2.5-coder-3b-instruct.Q4_K_M.gguf"

print(f"Size: {os.path.getsize(path) / (1024**3):.2f} GB")

Size: 1.80 GB


In [17]:
import os

folder = "/kaggle/working"

for f in [
    "qwen2.5_q4km.part_aa",
    "qwen2.5_q4km.part_ab",
    "qwen2.5_q4km.part_ac",
    "qwen2.5_q4km.part_ad"
]:
    path = os.path.join(folder, f)
    print(f, "→", round(os.path.getsize(path) / (1024**2), 1), "MB")

qwen2.5_q4km.part_aa → 500.0 MB
qwen2.5_q4km.part_ab → 500.0 MB
qwen2.5_q4km.part_ac → 500.0 MB
qwen2.5_q4km.part_ad → 340.5 MB


In [18]:
from IPython.display import FileLink, display

file_path = "/kaggle/working/qwen2.5_q4km.part_aa"

display(FileLink(file_path))

/kaggle/working/qwen2.5_q4km.part_aa

In [19]:
from IPython.display import FileLink, display

files = [
    "/kaggle/working/qwen2.5_q4km.part_aa",
    "/kaggle/working/qwen2.5_q4km.part_ab",
    "/kaggle/working/qwen2.5_q4km.part_ac",
    "/kaggle/working/qwen2.5_q4km.part_ad"
]

for f in files:
    display(FileLink(f))

/kaggle/working/qwen2.5_q4km.part_aa

/kaggle/working/qwen2.5_q4km.part_ab

/kaggle/working/qwen2.5_q4km.part_ac

/kaggle/working/qwen2.5_q4km.part_ad

In [20]:
import os
import shutil

src = "/kaggle/working"
files = [
    "qwen2.5_q4km.part_aa",
    "qwen2.5_q4km.part_ab",
    "qwen2.5_q4km.part_ac",
    "qwen2.5_q4km.part_ad"
]

for f in files:
    path = os.path.join(src, f)
    print(f, os.path.exists(path), 
          round(os.path.getsize(path)/(1024**2), 1) if os.path.exists(path) else "MISSING")

qwen2.5_q4km.part_aa True 500.0
qwen2.5_q4km.part_ab True 500.0
qwen2.5_q4km.part_ac True 500.0
qwen2.5_q4km.part_ad True 340.5


In [2]:
pip install kaggle

Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import os

GGUF_DIR = "/kaggle/working/qwen25_a2ui_lora/gguf_export"

print("=" * 70)
print("CHECKING GGUF EXPORT DIRECTORY")
print("=" * 70)

if os.path.exists(GGUF_DIR):
    print("Directory exists:", GGUF_DIR)
else:
    print("ERROR: directory does not exist!")

print("\nAll files currently inside export directory:\n")

for path in Path(GGUF_DIR).rglob("*"):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path}  -->  {size_mb:.1f} MB")

CHECKING GGUF EXPORT DIRECTORY
Directory exists: /kaggle/working/qwen25_a2ui_lora/gguf_export

All files currently inside export directory:

/kaggle/working/qwen25_a2ui_lora/gguf_export/model-00002-of-00002.safetensors  -->  1158.1 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/tokenizer_config.json  -->  0.0 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/tokenizer.json  -->  10.9 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/generation_config.json  -->  0.0 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/model-00001-of-00002.safetensors  -->  4727.9 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/config.json  -->  0.0 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/model.safetensors.index.json  -->  0.0 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/chat_template.jinja  -->  0.0 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/.cache/huggingface/.gitignore  -->  0.0 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export/.cache/huggingface/CACHEDIR.TAG  -->  0.0 MB
/kaggle/working/qwen25_

In [8]:
from pathlib import Path
import os

GGUF_DIR = "/kaggle/working/qwen25_a2ui_lora/gguf_export_gguf"

print("=" * 70)
print("CHECKING GGUF EXPORT DIRECTORY")
print("=" * 70)

if os.path.exists(GGUF_DIR):
    print("Directory exists:", GGUF_DIR)
else:
    print("ERROR: directory does not exist!")

print("\nAll files currently inside export directory:\n")

for path in Path(GGUF_DIR).rglob("*"):
    if path.is_file():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"{path}  -->  {size_mb:.1f} MB")

CHECKING GGUF EXPORT DIRECTORY
Directory exists: /kaggle/working/qwen25_a2ui_lora/gguf_export_gguf

All files currently inside export directory:

/kaggle/working/qwen25_a2ui_lora/gguf_export_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf  -->  1840.5 MB
/kaggle/working/qwen25_a2ui_lora/gguf_export_gguf/Modelfile  -->  0.0 MB


In [9]:
import os
import shutil

print("llama-quantize:", shutil.which("llama-quantize"))
print("llama.cpp:", shutil.which("llama.cpp"))

llama-quantize: None
llama.cpp: None


In [10]:
!find /kaggle/working -type f \( -name "llama-quantize" -o -name "convert_hf_to_gguf.py" \) 2>/dev/null

In [11]:
import os
import shutil
from pathlib import Path

GGUF_FILE = "/kaggle/working/qwen25_a2ui_lora/gguf_export_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf"

UPLOAD_DIR = "/kaggle/working/a2ui_gguf_upload"

# Create clean upload folder
if os.path.exists(UPLOAD_DIR):
    shutil.rmtree(UPLOAD_DIR)

os.makedirs(UPLOAD_DIR)

# Copy GGUF
destination = os.path.join(
    UPLOAD_DIR,
    os.path.basename(GGUF_FILE)
)

shutil.copy2(GGUF_FILE, destination)

print("GGUF copied successfully.")
print("Source:", GGUF_FILE)
print("Upload file:", destination)

size_gb = os.path.getsize(destination) / (1024 ** 3)

print(f"Size: {size_gb:.2f} GB")

GGUF copied successfully.
Source: /kaggle/working/qwen25_a2ui_lora/gguf_export_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf
Upload file: /kaggle/working/a2ui_gguf_upload/qwen2.5-coder-3b-instruct.Q4_K_M.gguf
Size: 1.80 GB


In [12]:
import kagglehub

DATASET = "rumanakhaisar/a2ui-finetuned-gguf"

print("Uploading to:")
print(DATASET)

kagglehub.dataset_upload(
    DATASET,
    UPLOAD_DIR,
    version_notes="Qwen2.5-Coder-3B A2UI fine-tuned Q4_K_M GGUF"
)

print("UPLOAD COMPLETE!")

Uploading to:
rumanakhaisar/a2ui-finetuned-gguf
Uploading Dataset https://kaggle.com/datasets/rumanakhaisar/a2ui-finetuned-gguf ...
Starting upload for file /kaggle/working/a2ui_gguf_upload/qwen2.5-coder-3b-instruct.Q4_K_M.gguf


Uploading: 100%|██████████| 1.93G/1.93G [00:44<00:00, 43.4MB/s]

Upload successful: /kaggle/working/a2ui_gguf_upload/qwen2.5-coder-3b-instruct.Q4_K_M.gguf (2GB)


HTTPError: 403 Client Error: Forbidden for url: https://api.kaggle.com/v1/datasets.DatasetApiService/CreateDatasetVersion